<a href="https://colab.research.google.com/github/Dwiynt/Beginer_python/blob/main/Lst_KSL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import ee
import geemap

def get_lst_for_month(year, month, roi):
    """
    Filters Landsat 8 Top-of-Atmosphere data for a given month,
    calculates Land Surface Temperature (LST) using Thermal Band 10,
    and returns a median image masked to the ROI.
    """
    start_date = f"{year}-{str(month).zfill(2)}-01"
    # Crude end-date calculation for monthly filtering
    if month == 12:
        end_date = f"{year+1}-01-01"
    else:
        end_date = f"{year}-{str(month+1).zfill(2)}-01"

    # Load Landsat 8 TOA collection
    collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
                  .filterBounds(roi)
                  .filterDate(start_date, end_date)
                  .filter(ee.Filter.lt('CLOUD_COVER', 30))) # Keep it relatively clear

    if collection.size().getInfo() == 0:
        return None

    median_scene = collection.median()

    # Calculate LST from Band 10 (Brightness Temperature in Kelvin to Celsius)
    # Formula: Celsius = Kelvin - 273.15
    lst_celsius = median_scene.select('B10').subtract(273.15).clip(roi)
    return lst_celsius

def main():
    # 1. Initialize GEE
    try:
        ee.Initialize()
        print("GEE Initialization Successful!")
    except Exception as e:
        print("Authenticating Earth Engine...")
        ee.Authenticate()
        # After authentication, ee.Initialize() still needs the project ID.
        # Replace 'your-project-id' with your actual Google Cloud Project ID.
        # You can find this in your Google Cloud Console.
        ee.Initialize(project='ee-dwiyanto')

    # 2. Define Area of Interest (ROI) using the center coordinates from your map
    # This creates a small bounding box buffer around your coordinate focal point in Sumatra
    lon, lat = 101.2853, -2.3432
    roi = ee.Geometry.Point([lon, lat]).buffer(50000).bounds()

    # 3. Create Map
    Map = geemap.Map(center=[lat, lon], zoom=9)

    # Standard visualization parameters for LST in Celsius
    lst_vis = {
        'min': 15,
        'max': 35,
        'palette': ['blue', 'green', 'yellow', 'orange', 'red']
    }

    # 4. Generate layers dynamically for the requested months
    months = [1, 3, 4, 5, 7, 8, 9, 10, 11]
    year = 2023 # Standard reference year with good Landsat 8 coverage

    print("\nCalculating LST layers directly from Landsat 8 data on-the-fly...")
    layers_added = 0

    for m in months:
        layer_name = f"LST 2023 - Month {str(m).zfill(2)}"
        print(f"Processing {layer_name}...")

        lst_img = get_lst_for_month(year, m, roi)

        if lst_img is not None:
            Map.addLayer(lst_img, lst_vis, layer_name)
            print(f"  ✅ Added to map.")
            layers_added += 1
        else:
            print(f"  ❌ No clear Landsat imagery found for month {m}.")

    # 5. Export interactive map to HTML
    if layers_added > 0:
        # No need for os or webbrowser in Colab for saving/opening local files
        # Just specifying the filename will save it to the default /content/ directory
        output_file = "lst_temperature_map.html"
        Map.to_html(filename=output_file)
        print(f"\n🎉 Success! The map file is saved as '{output_file}' in your Colab environment.")
        print("You can download it from the 'Files' tab on the left sidebar and open it in your web browser.")
    else:
        print("\n⏹️ No layers were generated. Check your internet connection or region bounds.")

if __name__ == "__main__":
    main()

Authenticating Earth Engine...

Calculating LST layers directly from Landsat 8 data on-the-fly...
Processing LST 2023 - Month 01...
  ❌ No clear Landsat imagery found for month 1.
Processing LST 2023 - Month 03...
  ❌ No clear Landsat imagery found for month 3.
Processing LST 2023 - Month 04...
  ✅ Added to map.
Processing LST 2023 - Month 05...
  ✅ Added to map.
Processing LST 2023 - Month 07...
  ❌ No clear Landsat imagery found for month 7.
Processing LST 2023 - Month 08...
  ❌ No clear Landsat imagery found for month 8.
Processing LST 2023 - Month 09...
  ✅ Added to map.
Processing LST 2023 - Month 10...
  ❌ No clear Landsat imagery found for month 10.
Processing LST 2023 - Month 11...
  ✅ Added to map.

🎉 Success! The map file is saved as 'lst_temperature_map.html' in your Colab environment.
You can download it from the 'Files' tab on the left sidebar and open it in your web browser.
